# Gesture model training (v2)

Trains the 1D-CNN gesture classifier from the `dataset/` CSV captures and
exports the int8 model for the Zephyr firmware.

Differences from the first notebook: fixed seeds, normalization statistics
computed on the training split only, a test set held out before any
preprocessing, early stopping, and a quantized-model verification step
before the firmware gets its blob.

In [ ]:
import glob
import os
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATASET_DIR = "../dataset"
CLASSES = ["A", "IDLE"]
COLUMNS = ["ax", "ay", "az", "gx", "gy", "gz"]
EXPECTED_SAMPLES = 256
SAMPLING_RATE_HZ = 100

# False: every capture goes to training, the field test on the board
# is the referee; True: honest holdout for methodology experiments
USE_HOLDOUT = False

TEST_SIZE = 0.2
VAL_SIZE = 0.2       # carved from the training half
EPOCHS = 40
BATCH_SIZE = 8
EARLY_STOP_PATIENCE = 10

## Dataset scan

Lists every class folder next to the dataset root; quarantine folders are
left alone on purpose.

In [ ]:
IGNORE_PREFIX = "quarantine_"

class_counts = {}
for entry in sorted(os.listdir(DATASET_DIR)):
    folder = os.path.join(DATASET_DIR, entry)
    if not os.path.isdir(folder) or entry.startswith(IGNORE_PREFIX):
        continue
    csv_files = sorted(glob.glob(os.path.join(folder, "*.csv")))
    if csv_files:
        class_counts[entry] = csv_files

for name, files in class_counts.items():
    print(f"{name}: {len(files)} captures")

## Validation and cleaning

Physical criteria per class; rejected windows move to
`dataset/quarantine_<class>/` instead of being deleted, so they can be
inspected later.

In [ ]:
# Peak angular rate and timing rules per class (rad/s, seconds)
RULES = {
    "A": {
        "min_peak_gyro": 3.0,
        "peak_time_min": 0.3,
        "peak_time_max": 1.8,
    },
    "IDLE": {
        "max_peak_gyro": 2.5,
    },
}


def validate_window(path, class_name):
    """Returns (ok, reason) for one CSV capture."""
    try:
        df = pd.read_csv(path, header=None, names=COLUMNS)
    except Exception as exc:
        return False, f"read error: {exc}"

    if len(df) != EXPECTED_SAMPLES:
        return False, f"invalid size ({len(df)}/{EXPECTED_SAMPLES} rows)"

    gyro_norm = np.sqrt(df["gx"] ** 2 + df["gy"] ** 2 + df["gz"] ** 2)
    max_gyro = float(gyro_norm.max())
    peak_time = float(gyro_norm.idxmax()) / SAMPLING_RATE_HZ

    rules = RULES.get(class_name)
    if rules is None:
        return True, "no rule for this class"

    if "min_peak_gyro" in rules and max_gyro < rules["min_peak_gyro"]:
        return False, f"peak too weak ({max_gyro:.2f} rad/s)"
    if "peak_time_min" in rules and peak_time < rules["peak_time_min"]:
        return False, f"gesture too early (peak at {peak_time:.2f} s)"
    if "peak_time_max" in rules and peak_time > rules["peak_time_max"]:
        return False, f"gesture late or incomplete (peak at {peak_time:.2f} s)"
    if "max_peak_gyro" in rules and max_gyro > rules["max_peak_gyro"]:
        return False, f"stray motion ({max_gyro:.2f} rad/s)"

    return True, "ok"


summary = {}
for class_name, files in class_counts.items():
    quarantine = os.path.join(DATASET_DIR, f"quarantine_{class_name}")
    os.makedirs(quarantine, exist_ok=True)
    kept, moved = [], 0
    for path in files:
        ok, reason = validate_window(path, class_name)
        if ok:
            kept.append(path)
        else:
            print(f"[REJECTED] {os.path.basename(path)}: {reason}")
            shutil.move(path, os.path.join(quarantine, os.path.basename(path)))
            moved += 1
    summary[class_name] = kept
    print(f"{class_name}: {len(kept)} kept, {moved} quarantined")

## Split before anything else

The test set is carved out first and never touched again: no statistics,
no normalization, no tuning. Class balance is reported once.

In [ ]:
X, y = [], []
for class_idx, class_name in enumerate(CLASSES):
    for path in summary[class_name]:
        df = pd.read_csv(path, header=None, names=COLUMNS)
        if len(df) == EXPECTED_SAMPLES:
            X.append(df.values)
            y.append(class_idx)

X = np.array(X, dtype=np.float32)   # (N, 256, 6)
y = np.array(y, dtype=np.int32)

print(f"total: {X.shape[0]} samples {X.shape[1:]}")
for class_idx, class_name in enumerate(CLASSES):
    print(f"  {class_name}: {int(np.sum(y == class_idx))}")

if USE_HOLDOUT:
    X_rest, X_test, y_rest, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_rest, y_rest, test_size=VAL_SIZE, random_state=SEED, stratify=y_rest
    )
    print(f"train/val/test: {len(X_train)}/{len(X_val)}/{len(X_test)}")
else:
    X_train, y_train = X, y
    X_test = X_val = y_test = y_val = None
    print("holdout off: every sample goes to training")

## Normalization

Per-channel z-score computed on the **training split only**; the same
constants travel to the firmware and must never be recomputed from data the
test set influenced.

In [ ]:
mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True) + 1e-6

X_train = (X_train - mean) / std
if USE_HOLDOUT:
    X_val = (X_val - mean) / std
    X_test = (X_test - mean) / std

print("float norm_mean[6] = {" + ", ".join(f"{v:.6f}f" for v in mean.flatten()) + "};")
print("float norm_std[6]  = {" + ", ".join(f"{v:.6f}f" for v in std.flatten()) + "};")

## Model

Spec topology: Conv1D(16, 5) + pool, Conv1D(32, 3) + pool, global average
pooling, Dense(16), softmax over the classes. Early stopping watches
validation loss and restores the best weights.

In [ ]:
model = models.Sequential([
    layers.Input(shape=(EXPECTED_SAMPLES, len(COLUMNS))),
    layers.Conv1D(16, kernel_size=5, padding="same", activation="relu"),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(32, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling1D(pool_size=2),
    layers.GlobalAveragePooling1D(),
    layers.Dense(16, activation="relu"),
    layers.Dense(len(CLASSES), activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss" if USE_HOLDOUT else "loss",
        patience=EARLY_STOP_PATIENCE,
        restore_best_weights=True)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val) if USE_HOLDOUT else None,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=2,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(history.history["loss"], label="train")
if USE_HOLDOUT:
    ax1.plot(history.history["val_loss"], label="val")
ax1.set_title("loss")
ax1.legend()
ax2.plot(history.history["accuracy"], label="train")
if USE_HOLDOUT:
    ax2.plot(history.history["val_accuracy"], label="val")
ax2.set_title("accuracy")
ax2.legend()
plt.show()

## Float model: honest report

Accuracy and confusion matrix on the untouched test split.

In [ ]:
if not USE_HOLDOUT:
    print("no holdout: the field test on the board is the referee")
else:
    y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
    test_acc = (y_pred == y_test).mean()
    print(f"float model test accuracy: {test_acc * 100:.1f}%")

    cm = confusion_matrix(y_test, y_pred, labels=range(len(CLASSES)))
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot()
    plt.show()

## Int8 conversion (TFLM)

Full integer quantization; the representative dataset comes from the
training split only. Dense layers are kept per-tensor because this TFLM
revision does not support per-channel quantization on FULLY_CONNECTED.

In [ ]:
def representative_dataset():
    for sample in X_train:
        yield [np.expand_dims(sample, axis=0).astype(np.float32)]


converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
converter._experimental_enforce_full_integer_quantization = True
converter._experimental_disable_per_channel_quantization_for_dense_layers = True

tflite_model = converter.convert()

with open("model_gesture_int8.tflite", "wb") as fh:
    fh.write(tflite_model)
print(f"int8 model: {len(tflite_model)} bytes (~{len(tflite_model)//1024} KB)")

## Quantized model: the gate before the firmware

Runs the TFLite model on the test split and compares it with the float
model. Quantization damage shows up here, on the desk, not on the board.

In [ ]:
eval_X = X_test if USE_HOLDOUT else X_train[:20]
eval_y = y_test if USE_HOLDOUT else y_train[:20]
eval_name = "test" if USE_HOLDOUT else "train (quantization drift only)"

interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]
print(f"input : {inp['dtype']}, scale={inp['quantization'][0]:.8f}, "
      f"zero_point={inp['quantization'][1]}")
print(f"output: {out['dtype']}, scale={out['quantization'][0]:.8f}, "
      f"zero_point={out['quantization'][1]}")

correct = 0
max_prob_delta = 0.0
for sample, label in zip(eval_X, eval_y):
    quant = np.clip(np.round(sample / inp['quantization'][0])
                    + inp['quantization'][1], -128, 127).astype(np.int8)
    interpreter.set_tensor(inp['index'], quant.reshape(inp['shape']))
    interpreter.invoke()
    probs_int8 = interpreter.get_tensor(out['index'])[0].astype(np.float32)
    probs = (probs_int8 - out['quantization'][1]) * out['quantization'][0]
    correct += int(probs.argmax() == label)
    max_prob_delta = max(max_prob_delta,
                         abs(probs[label] - model.predict(sample[None],
                                                          verbose=0)[0][label]))

print(f"int8 accuracy on {eval_name}: {correct / len(eval_X) * 100:.1f}%")
print(f"largest probability drift vs float: {max_prob_delta:.4f}")

## Firmware exports

`model_data.h` is the int8 model blob;
`model_params.h` carries the z-score and quantization constants, generated from the
converted model itself. Both land in `model/`.


In [ ]:
inp_q = inp["quantization"]
out_q = out["quantization"]

with open("../model/model_data.h", "w") as fh:
    fh.write("#ifndef MODEL_DATA_H_\n#define MODEL_DATA_H_\n\n")
    fh.write("#include <zephyr/toolchain.h>\n")
    fh.write("#include <zephyr/types.h>\n\n")
    fh.write(f"const unsigned int g_model_len = {len(tflite_model)};\n")
    fh.write("const unsigned char g_model[] __aligned(4) = {\n  ")
    for idx, byte in enumerate(tflite_model):
        fh.write(f"0x{byte:02x}, ")
        if (idx + 1) % 12 == 0:
            fh.write("\n  ")
    fh.write("\n};\n\n#endif /* MODEL_DATA_H_ */\n")

with open("../model/model_params.h", "w") as fh:
    fh.write("#ifndef MODEL_PARAMS_H_\n#define MODEL_PARAMS_H_\n\n")
    fh.write("#include <stdint.h>\n\n")
    fh.write("/* Generated by notebook_jupiter/train_v2.ipynb, "
             "do not edit by hand */\n\n")
    fh.write("/* Per-channel z-score of the training split */\n")
    fh.write("static const float norm_mean[6] = {"
             + ", ".join(f"{v:.6f}f" for v in mean.flatten()) + "};\n")
    fh.write("static const float norm_std[6]  = {"
             + ", ".join(f"{v:.6f}f" for v in std.flatten()) + "};\n\n")
    fh.write("/* Input quantization: int8 = round(x_norm / kInputScaleInv) "
             "+ kInputZeroPoint */\n")
    fh.write(f"static const float kInputScaleInv = {1.0 / inp_q[0]:.6f}f;\n")
    fh.write(f"static const int32_t kInputZeroPoint = {int(inp_q[1])};\n\n")
    fh.write("/* Output de-quantization: P(class) = (q - kOutputZeroPoint) "
             "* kOutputScale */\n")
    fh.write(f"static const float kOutputScale = {out_q[0]:.8f}f;\n")
    fh.write(f"static const int32_t kOutputZeroPoint = {int(out_q[1])};\n\n")
    fh.write("#endif /* MODEL_PARAMS_H_ */\n")

print("model/model_data.h and model/model_params.h written")